# style-tuned-llm: QLoRA training on a free Colab T4

Fine-tunes Qwen2.5-1.5B-Instruct to rewrite complex text in plain English.

This is run 2. Its training data also rejects pairs that drop a number, and its
adapter goes to a separate `-v2` Hub repo, so run 1 is kept for comparison.

**Before you start:** choose *Runtime > Change runtime type > T4 GPU*, then run the
cells in order. Training is capped at 45 minutes. The whole notebook needs no paid API.

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: choose Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Get the code and install pinned packages

In [ ]:
%cd /content
!git clone https://github.com/arbablatif/style-tuned-llm.git
%cd /content/style-tuned-llm
!pip install -q -r requirements-colab.txt

## 3. Log in to Hugging Face

Use a token with **write** access, from https://huggingface.co/settings/tokens.
The token is not stored in this notebook.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import whoami
from src.config import ADAPTER_REPO_NAME
HUB_REPO = f"{whoami()['name']}/{ADAPTER_REPO_NAME}"
print("The adapter will be pushed to:", HUB_REPO)

## 4. Prepare the data

In [ ]:
!python -m src.prepare_data

## 5. Train and push the adapter

If you see `CUDA out of memory`, add `--batch-size 4` and run this cell again.

In [ ]:
!python -m src.train --push-repo {HUB_REPO}

## 6. Check for overfitting

Validation loss should fall and then level off. If it rises while training
loss keeps falling, the model is overfitting.

In [ ]:
import json
log = json.load(open("results/train_log.json"))
print({k: v for k, v in log.items() if k != "log_history"})
train = {e["step"]: e["loss"] for e in log["log_history"] if "loss" in e}
for entry in log["log_history"]:
    if "eval_loss" in entry:
        step = entry["step"]
        print(f"step {step:>5}  train {train.get(step, float('nan')):.3f}  "
              f"validation {entry['eval_loss']:.3f}")

## 7. Evaluate base vs fine-tuned

In [ ]:
!python -m src.evaluate --adapter outputs/adapter

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results/examples.md", encoding="utf-8").read()))

## 8. Download the results

**Do this before trying the demo.** Colab deletes these files when the session ends.
Save `style_tuned_results.zip`, then send it back or unzip it over your local repo.

In [ ]:
!zip -q -r style_tuned_results.zip results -x "results/v1/*"
from google.colab import files
files.download("style_tuned_results.zip")

## 9. Try the demo (optional)

Open the public `gradio.live` link printed below. Press the stop button on this
cell when you finish. Anyone with the link can use the demo while it runs.

In [ ]:
!python app.py --adapter outputs/adapter --share